# 4. Model — PDF §7.4, páginas 8–9

Se comparan Regresión Logística y Random Forest. Cada algoritmo aprende los objetivos binario y multietiqueta mediante One-vs-Rest. Las predicciones no se generan mediante expresiones regulares.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Abrir desde avance_semma o su carpeta notebooks'
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image, Markdown


## Entrenamiento y validación

Se prueban dos configuraciones por algoritmo y umbrales 0,30–0,70. Se selecciona por F1 macro de las diez categorías con evidencia en validación; F1 binario y tiempo resuelven empates. No se consulta test para escoger. El pipeline se serializa antes de evaluar test.

In [2]:
from src.experiment import model_phase
comparison = model_phase()
display(pd.read_csv(ROOT / 'results/validation_search.csv'))
display(pd.DataFrame(comparison))

Trained lr_C0.5 in 0.5s


Trained lr_C2 in 0.6s


Trained rf_100 in 3.5s


Trained rf_150_depth20 in 4.8s


,candidate,family,threshold,fit_seconds,parameters,binary_precision,binary_recall,binary_f1,category_precision_macro,category_recall_macro,category_f1_macro,category_f1_micro,hamming_loss,subset_accuracy
0,rf_150_depth20,random_forest,0.3,4.759499,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""class_w...",0.827586,1.000000,0.905660,0.789213,1.000000,0.845987,0.940217,0.006509,0.946746
1,rf_100,random_forest,0.5,3.488205,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""class_w...",0.970238,0.970238,0.970238,0.691873,0.796186,0.669313,0.910180,0.008876,0.926036
2,lr_C2,logistic_regression,0.5,0.615993,"{""C"": 2, ""class_weight"": ""balanced"", ""dual"": f...",0.877095,0.934524,0.904899,0.623106,0.676318,0.625509,0.860606,0.013609,0.878698
3,lr_C2,logistic_regression,0.3,0.615993,"{""C"": 2, ""class_weight"": ""balanced"", ""dual"": f...",0.688525,1.000000,0.815534,0.459466,0.994340,0.597371,0.806763,0.023669,0.804734
4,lr_C0.5,logistic_regression,0.4,0.534902,"{""C"": 0.5, ""class_weight"": ""balanced"", ""dual"":...",0.652344,0.994048,0.787736,0.447410,0.982804,0.575802,0.787129,0.025444,0.798817
5,rf_100,random_forest,0.6,3.488205,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""class_w...",0.987500,0.940476,0.963415,0.638876,0.566655,0.566050,0.883281,0.010947,0.902367
6,rf_100,random_forest,0.4,3.488205,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""class_w...",0.932961,0.994048,0.962536,0.496218,0.953396,0.562951,0.888889,0.012130,0.940828
7,lr_C0.5,logistic_regression,0.5,0.534902,"{""C"": 0.5, ""class_weight"": ""balanced"", ""dual"":...",0.850000,0.910714,0.879310,0.569284,0.536581,0.534672,0.788462,0.019527,0.825444
8,lr_C2,logistic_regression,0.4,0.615993,"{""C"": 2, ""class_weight"": ""balanced"", ""dual"": f...",0.818627,0.994048,0.897849,0.444215,0.825463,0.527120,0.833333,0.018343,0.852071
9,rf_150_depth20,random_forest,0.4,4.759499,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""class_w...",0.949153,1.000000,0.973913,0.506353,0.571123,0.519106,0.938416,0.006213,0.949704


,family,candidate,threshold,train_category_f1_macro,validation_category_f1_macro
0,random_forest,rf_150_depth20,0.3,0.888438,0.845987
1,logistic_regression,lr_C2,0.5,0.838527,0.625509


In [3]:
display(pd.Series(json.loads((ROOT / 'results/model_selection.json').read_text(encoding='utf-8'))))
print('Artefactos:', [p.name for p in (ROOT / 'models').glob('*.joblib')])

candidate                                                          rf_150_depth20
family                                                              random_forest
threshold                                                                     0.3
criterion                       Maximum validation category F1 macro; binary F...
validation_category_f1_macro                                             0.845987
test_used_for_selection                                                     False
refit_on_validation                                                         False
dataset_sha256                  0c42cd801c75cff46b688be800b966f0de4266c17ea515...
dtype: object

Artefactos: ['logistic_regression.joblib', 'modelo_seleccionado.joblib', 'preprocessor.joblib', 'random_forest.joblib']


## Inferencia posterior

El artefacto seleccionado incluye vectorización, clasificadores, catálogo y umbral. Su entrada es código fuente, sin CWE ni categoría conocida. Un resultado incierto deja la categoría pendiente de revisión. Los scores no están calibrados y no representan severidad. Las categorías sin datos están declaradas como no evaluadas.